<h2 style="font-size:40px; text-align:center">Scraped from Ambitions (website)</h2>

In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
from tqdm.auto import tqdm
from fake_useragent import UserAgent
import time

In [3]:
ua= UserAgent()
name=[]
rating=[]
reviews=[]
salary=[]
jobs=[]
interviews=[]

In [4]:
headers={
    "User-Agent": ua.random
}
base_url = "https://www.ambitionbox.com/list-of-companies?campaign=desktop_nav&page={}"

for i in tqdm(range(1, 501), desc="Scraping Pages"):

    try:
        url = base_url.format(i)
        response = requests.get(url, headers=headers,)

        if response.status_code != 200:
            continue

        soup = BeautifulSoup(response.text, "lxml")
        companies = soup.find_all("div", class_="companyCardWrapper")

        for j in companies:

            # --- Name ---
            name_tag = j.find("h2")
            name.append(name_tag.text.strip() if name_tag else None)

            # --- Rating ---
            rating_tag = j.find("div", style="height:auto;padding-bottom:1px;")
            rating.append(rating_tag.text.strip() if rating_tag else None)

            # --- Action counts ---
            counts = j.find_all("span", class_="companyCardWrapper__ActionCount")

            reviews.append(counts[0].text.strip() if len(counts) > 0 else None)
            salary.append(counts[1].text.strip() if len(counts) > 1 else None)
            interviews.append(counts[2].text.strip() if len(counts) > 2 else None)
            jobs.append(counts[3].text.strip() if len(counts) > 3 else None)

        time.sleep(1)  # polite delay

    except Exception as e:
        print(f"Error on page {i}: {e}")


Scraping Pages:   0%|          | 0/500 [00:00<?, ?it/s]

In [5]:
len(name)

10000

In [6]:
d={'name':name, 'jobs': jobs, 'interviews':interviews, 'salary':salary, 'rating':rating, 'reviews':reviews}
df=pd.DataFrame(d)

In [7]:
df.head()

,name,jobs,interviews,salary,rating,reviews
0,TCS,2.8k,11.9k,9.8L,3.3,1.1L
1,Accenture,41.6k,9.2k,6.5L,3.7,71.7k
2,Wipro,7k,6.7k,4.8L,3.6,63.8k
3,Cognizant,661,6.3k,6L,3.6,60.1k
4,Capgemini,2.2k,5.5k,4.8L,3.7,51.7k


In [8]:
df.shape

(10000, 6)

In [9]:
#df.to_csv(r'./CSV_by_WebScraping/Companies.csv', index=False)